In [2]:
import example_loader as el
import gurobipy as gp
import gurobi_utils as gu
import miplib_loader as ml
import numpy as np
import plot_utils as pu

In [3]:
# configure the backend for matplotlib
# this one should allow zoom:
%matplotlib widget
# to make that work you need: "pip install ipympl" and run "jupyter nbextension enable --py widgetsnbextension"

# this will work without the above dependencies but won't allow zoom
# %matplotlib inline

# this option may work whenever they fix bugs in mpld3
# import mpld3
# mpld3.enable_notebook()

In [ ]:
# A function to create cuts given a target point
def add_balas_ball_cut(relaxed: gp.Model, target, integer_vars, integer_idx, plotter, bounds_tightening=False):
    # for each column in the tableau
    # construct a sparse vector for it
    # get the length of that vector via norm1 (plus 1 if we're an int column)
    # add our cut: sum_j(x_j/a_j)
    
    norm = 2
    current = integer_vars.X
    radius = np.linalg.norm(current - target, norm)
    if radius <= relaxed.params.FeasibilityTol:
        return False  # TODO: tolerance should apply to each component individually?
    
    if len(current) < 7:
        print("   Gap to target:", radius, ":", current, "to", target)
    else:
        print("   Gap to target:", radius, "Score:", relaxed.getObjective().getValue())
    if plotter is not None:
        plotter.add_ball(radius)

    variables = relaxed.getVars()  # TODO: pass this in as it's expensive?
    constraints = relaxed.getConstrs()  # wish we didn't have to use this one
    
    basis = gu.read_basis(relaxed)
    tableau, col_to_var, negated_rows = gu.read_tableau(relaxed, basis, extra_rows=1)
    negated_vars = [basis[nr] for nr in negated_rows]
    
    # drop the rows of non-integer variables:
    to_drop = [i for i, base in enumerate(basis) if base not in integer_idx]
    tableau = np.delete(tableau, to_drop, axis=0)  # TODO: don't even bother to read them in
    basis = np.delete(basis, to_drop)

    # Conforti has negative vectors with 1 at row=col, with the rest negated.
    # However, empirically, it seems that the opposite is what we really want (gurobi-specific or ineq. standardization issue)
    int_cols = [i for i, c in enumerate(col_to_var) if c in integer_idx]
    tableau[-1, int_cols] = -1  # use our extra row to store the col==row -> 1
    lengths = np.linalg.norm(tableau, norm, axis=0)
    lengths /= radius

    if bounds_tightening:  # assuming this is only true on a feasible target
        tol = relaxed.params.IntFeasTol
        for i, base in enumerate(basis):
            all_same_sign = np.all(tableau[i, :] > -tol) or np.all(tableau[i, :] < tol)  # TODO: find a more efficient call for this
            if all_same_sign and target[integer_idx[base]] > current[integer_idx[base]]:
                new_con = relaxed.addConstr(variables[base] >= target[integer_idx[base]])
                print("  Tightened", variables[base].VarName, ">=", target[integer_idx[base]])
                if plotter is not None:
                    relaxed.update()
                    plotter.add_constraint(new_con, color='xkcd:blue green')
            elif all_same_sign and target[integer_idx[base]] < current[integer_idx[base]]:
                new_con = relaxed.addConstr(-variables[base] >= -target[integer_idx[base]])
                print("  Tightened", variables[base].VarName, "<=", target[integer_idx[base]])
                if plotter is not None:
                    relaxed.update()
                    plotter.add_constraint(new_con, color='xkcd:blue green')
    
    relaxed.update()
    def find_variable(index):
        if index < len(variables):
            # handle inverted variables (SCIP and Gurobi both have this silliness)
            if variables[index].VBasis == -2:  # not yet sure this is correct
                return variables[index].X - variables[index]
            assert variables[index].VBasis == -1  # not handling -3 yet
            return variables[index]
        # if only gurobi gave us access to their slack variables...
        # instead, we have to solve for it:
        cons_idx = index - len(variables)
        constraint = constraints[cons_idx]
        lhs, sense, rhs = relaxed.getRow(constraint), constraint.Sense, constraint.RHS
        if sense == '<':
             return rhs - lhs
        elif sense == '>':
            return lhs - rhs
        else:
            return 0.0  # is this right for equality constraints?
    
    summed_terms = gp.quicksum(lengths[i] * find_variable(j) for i, j in enumerate(col_to_var))
    new_con = relaxed.addConstr(summed_terms >= 1)

    print("   Moved:", 1.0 / np.linalg.norm(lengths, norm))
    
    # what does it mean to take a relaxed point in full space and compare only its want-to-be-integer components to another?
    if plotter is not None:
        relaxed.update()
        plotter.add_constraint(new_con)
    return True

def run_cuts_to_relaxed_sol(instances, cut_function, bounds_tightening=True, loops=8):
    for instance in instances:
        model: gp.Model = instance.as_gurobi_model()
        print("Running model:", model.ModelName)
        model.params.LogToConsole = 0
        plotter = pu.create(model)

        exact_model = model.copy()
        exact_model.update()
        exact_vars = gp.MVar.fromlist([v for v in exact_model.getVars() if v.VType != 'C'])
        z = exact_model.addMVar(exact_vars.shape, lb=-gp.GRB.INFINITY)
        exact_model.setObjective(z.sum(), gp.GRB.MINIMIZE)
        
        int_vars, int_idx = gu.relax_int_or_bin_to_continuous(model)
        if bounds_tightening:
            gu.standardize_lt_to_gt(model)
            cons1 = gu.standardize_ub_to_constr(model)
            cons2 = gu.standardize_lb_to_constr(model)
            model.update()
            if plotter is not None:
                for con in cons1 + cons2:
                    plotter.add_constraint(con, color='xkcd:steel blue')

        for i in range(loops):
            model.optimize()
            if model.Status != gp.GRB.OPTIMAL:
                print("   FAILED! Status:", model.Status)
                break

            relaxed_x = int_vars.X
            ca = exact_model.addConstr(z >= exact_vars - relaxed_x)
            cb = exact_model.addConstr(z >= relaxed_x - exact_vars)
            exact_model.optimize()
            assert exact_model.Status == gp.GRB.OPTIMAL

            if not cut_function(model, exact_vars.X, int_vars, int_idx, plotter, bounds_tightening):
                print("   Final score of relaxed:", model.getObjective().getValue())
                break

            exact_model.remove(ca)
            exact_model.remove(cb)
        print("   Known best score:", instance.score if instance.known_optimum else "unknown")    
        if plotter is not None:
            plotter.render()

# test the cuts on simple examples:
run_cuts_to_relaxed_sol(list(el.get_instances().values()), add_balas_ball_cut, bounds_tightening=True)

In [9]:
miplib_instances = ml.get_instances()
miplib_subset = [miplib_instances['air05'], miplib_instances['markshare2']]  # mas76
for instance in miplib_subset:
    mi = instance.as_gurobi_model()
    mi.params.Heuristics = 0
    mi.params.NoRelHeurTime = 2.0
    mi.params.DisplayInterval = 1
    mi.optimize()
    print("Expected optimum:", instance.score)

Read MPS format model from file mip2017_benchmark/revised-submissions/miplib2010_publically_available/instances/air05.mps.gz
Reading time = 0.02 seconds
air05: 426 rows, 7195 columns, 52121 nonzeros
Set parameter Heuristics to value 0
Set parameter NoRelHeurTime to value 2
Set parameter DisplayInterval to value 1
Gurobi Optimizer version 10.0.3 build v10.0.3rc0 (linux64)

CPU model: Intel(R) Core(TM) i7-8750H CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 6 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 426 rows, 7195 columns and 52121 nonzeros
Model fingerprint: 0xa71082fe
Variable types: 0 continuous, 7195 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [4e+01, 3e+03]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]
Presolve removed 89 rows and 1070 columns
Presolve time: 0.18s
Presolved: 337 rows, 6125 columns, 36081 nonzeros
Variable types: 0 continuous, 6125 integ

In [23]:
import jsplib_loader as jl
jsplib_instances = jl.get_instances()
jsplib_subset = [jsplib_instances['abz7']]
for instance in jsplib_subset:
    mi = instance.as_gurobi_model()
    mi.params.Heuristics = 0
    mi.params.NoRelHeurTime = 100.0
    mi.params.DisplayInterval = 1
    mi.params.Cuts = 0
    mi.params.CutPasses = 0
    mi.optimize()
    print("Expected optimum:", instance.score)

Set parameter AggFill to value 10
Set parameter GomoryPasses to value 1
Set parameter Heuristics to value 0
Set parameter NoRelHeurTime to value 100
Set parameter DisplayInterval to value 1
Set parameter Cuts to value 0
Set parameter CutPasses to value 0
Gurobi Optimizer version 10.0.3 build v10.0.3rc0 (linux64)

CPU model: Intel(R) Core(TM) i7-8750H CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 6 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 6000 rows, 6301 columns and 17700 nonzeros
Model fingerprint: 0xe09341db
Variable types: 301 continuous, 6000 integer (6000 binary)
Coefficient statistics:
  Matrix range     [1e+00, 7e+03]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+01, 7e+03]
Presolve removed 0 rows and 3150 columns
Presolve time: 0.02s
Presolved: 6000 rows, 3151 columns, 17700 nonzeros
Variable types: 301 continuous, 2850 integer (2850 binary)
Starting NoRel heuristic
Found phas

In [ ]:
import importlib
importlib.reload(gu)

miplib_instances = ml.get_instances()
miplib_subset = [miplib_instances['air05'], miplib_instances['markshare2']]  # mas76
run_cuts_to_relaxed_sol(miplib_subset[0:1], add_balas_ball_cut)

In [ ]:
import importlib
importlib.reload(gu)

import jsplib_loader as jl
jsplib_instances = jl.get_instances()
jsplib_subset = [jsplib_instances['abz5']]
run_cuts_to_relaxed_sol(jsplib_subset, add_balas_ball_cut, bounds_tightening=True, loops=20)
# jsplib_subset[0].as_gurobi_model().optimize()

In [26]:
# run comparisons:
def exit_early(model: gp.Model, where):
    if where == gp.GRB.Callback.MIPNODE:
        if model.cbGet(gp.GRB.Callback.MIPNODE_NODCNT) > 0:
            dualBound = model.cbGet(gp.GRB.Callback.MIPNODE_OBJBND)
            primalBound = model.cbGet(gp.GRB.Callback.MIPNODE_OBJBST)
            print("   Gurobi's best dual bound:", dualBound, ", best primal:", primalBound)
            model.terminate()

def compare_cuts(instances):
    for instance in instances:
        run_cuts_to_relaxed_sol([instance], add_balas_ball_cut, loops=15, bounds_tightening=True)
        print("Attempting Gurobi's aggressive cuts:")
        mi = instance.as_gurobi_model()
        mi.params.Cuts = 3
        mi.params.CutPasses = 10
        mi.params.LogToConsole = 0
        mi.optimize(exit_early)

        print("Attempting Gurobi's NoRel:")
        mi = instance.as_gurobi_model()
        mi.params.Heuristics = 0
        mi.params.NoRelHeurTime = 12.0
        mi.params.Cuts = 0
        mi.params.CutPasses = 0
        mi.params.LogToConsole = 0
        mi.optimize(exit_early)
        
        print("Expected optimum:", instance.score)    

import jsplib_loader as jl
jsplib_instances = jl.get_instances()
jsplib_subset = [jsplib_instances['abz7']]

miplib_instances = ml.get_instances()
miplib_subset = [miplib_instances['air05'], miplib_instances['markshare2']]  # mas76

compare_cuts(miplib_subset)

Read MPS format model from file mip2017_benchmark/revised-submissions/miplib2010_publically_available/instances/air05.mps.gz
Reading time = 0.02 seconds
air05: 426 rows, 7195 columns, 52121 nonzeros
Running model: air05
   Relaxed 7195 variables on air05
   Negated 0 constraints on air05
   Standardized 7195 upper bounds to be constraints
   Standardized 0 lower bounds to be constraints
   Gap to target: 5.313768418382037 Score: 25877.609267851407
   Moved: 0.0007453700312680283
   Gap to target: 5.31922708737515 Score: 25877.688275999524
   Moved: 0.003278275494877523
   Gap to target: 5.290767359209419 Score: 25878.090828002296
   Moved: 0.003210526799461707
   Gap to target: 5.399211392630394 Score: 25878.637998116068
   Moved: 0.0014136349602484582
   Gap to target: 5.333559620242232 Score: 25878.784794735726
   Moved: 0.006162116582871152
   Gap to target: 5.3364870387314625 Score: 25879.198407160777
   Moved: 0.0046327533907498395
   Gap to target: 5.34362354172555 Score: 25879.5